# Lesson 2 — Score, Noul, and picking the right primitive

Lesson 1 covered System One models, `state`, and `Choice`. This lesson covers the other two question types — `Score` and `Noul` — and how to tell which one a given judgment actually needs.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/primitives/score (click to expand)</summary>

# Score

> A Score is a System One question type for rating content against ordered, descriptive levels. The answer includes a score, a probability for each level, and confidence.

Use a Score when the answer is a position on a spectrum you can describe in steps. For example, how severe a bug is, how happy a customer is, or how much Python experience a candidate has. If the answer is one of a fixed set of options with no order between them, use a [Choice](https://docs.typesafe.ai/primitives/choice). If it's a yes or no, use a [Noul](https://docs.typesafe.ai/primitives/noul). [Choose a question type](https://docs.typesafe.ai/primitives#choose-a-question-type) compares all three.

A Score answer is a position along your levels in `score`, which can fall between two levels. The model also returns a probability for every level in `probabilities`, and a `confidence` value for the answer.

Example Score questions:

```
"How severe is the bug being reported?"
  → 0: Cosmetic; no impact to functionality
  → 1: Broken or degraded feature, but workaround exists
  → 2: Blocking issue; no workaround exists

"How formal is this outfit based on the description"
  → 0: gym clothes
  → 1: casual
  → 2: business casual
  → 3: formal
  → 4: black tie

"How relevant is this candidate's experience to the job posting"
  → 0: completely unrelated
  → 1: adjacent field
  → 2: some direct experience
  → 3: deep, direct experience
```

## Request structure

The POST request body to the [TypeSafe API](https://docs.typesafe.ai/api) has the same three top-level fields as any other question type: `state`, which is the content to evaluate; `model`; and `questions`. Each Score question has the following fields:

- `type`: Always `"score"`.
- `instructions`: The question the model answers. What it's rating.
- `criteria`: An ordered array of level descriptions, from the low end of the scale to the high end. Needs at least two levels and takes up to 10.

Below is a request where the state is a bug report and the question is how severe the bug is:

```json
{
  "state": "The export button crashes the settings page in Safari. It works in Chrome, but a few of our customers only use Safari.",
  "questions": {
    "bug_severity": {
      "type": "score",
      "instructions": "How severe is the reported issue?",
      "criteria": [
        "Cosmetic; no impact to functionality",
        "Broken or degraded feature, but workaround exists",
        "Blocking issue; no workaround exists"
      ]
    }
  }
}
```

You choose the question id, `bug_severity` in this case. This id is not sent to the model. The answer is returned under the same id.

### Levels

Each entry in `criteria` is a level: one point on the spectrum of possible answers, described in words. A level's number is its position in the `criteria` array, starting at 0, so the three entries above are levels 0, 1 and 2. The order of the array is the numbering.

The model gets the descriptions and nothing else, and each level is judged on its own against the state.

The `score` in the response is a position on the levels spectrum. For a three-level scale it runs from 0 to 2, and it can land between two levels.

Our [client SDKs](https://docs.typesafe.ai/sdk) provide typed questions. In Python, the same question is a `Score`:

```python
from typesafe_sdk import Score, TypeSafeClient

with TypeSafeClient() as client:
    response = client.system_one(
        state="The export button crashes the settings page in Safari. It works in Chrome, but a few of our customers only use Safari.",
        questions={
            "bug_severity": Score(
                instructions="How severe is the reported issue?",
                criteria=[
                    "Cosmetic; no impact to functionality",
                    "Broken or degraded feature, but workaround exists",
                    "Blocking issue; no workaround exists",
                ],
            ),
        },
    )

    print(response.answers["bug_severity"].score)
```

Use the `system_one` method or the `https://api.typesafe.ai/v1/systemone` endpoint to call a System One model. The `model` field selects which model handles the request. [How to build with TypeSafe](https://docs.typesafe.ai/concepts/how-to-build-with-system-one) covers where in your code to call it.

> **Note:** `instructions` and each level in `criteria` can be a string, an object, or an array. Start with strings. Use an object when a level needs a description plus a few example situations.

## Response structure

The response has one entry in `answers` per question, under the ids from the request. This is the response to the example request above:

```json
{
  "model": "jev-latest",
  "answers": {
    "bug_severity": {
      "type": "score",
      "score": 1.3,
      "confidence": 0.54,
      "legend": {
        "0": "Cosmetic; no impact to functionality",
        "1": "Broken or degraded feature, but workaround exists",
        "2": "Blocking issue; no workaround exists"
      },
      "probabilities": {
        "0": 0.0,
        "1": 0.7,
        "2": 0.3
      }
    }
  },
  "usage": {
    "input_tokens": 332,
    "output_tokens": 18
  }
}
```

Each Score answer has five values:

- `type`: The type of TypeSafe question.
- `probabilities`: The probability of each level, keyed by level number as a string. The sum of all values is 1.
- `score`: The position on the level number line, from 0 to the top level number, which is 2 here. It's each level number multiplied by its probability, added up: 0 × 0.0 + 1 × 0.70 + 2 × 0.30 = 1.30.
- `legend`: Each level number mapped back to its description.
- [`confidence`](https://docs.typesafe.ai/confidence): A number from 0 to 1 computed from how `probabilities` is spread. A single peak on one level means high confidence. Probability spread over several levels means low confidence.

A score of 1.30 means mostly level 1 with some weight on level 2. That matches the report: the export is broken, and switching to Chrome is a workaround for most customers, but not for the ones who only use Safari. The model puts 0.70 on "workaround exists" and 0.30 on "no workaround", and confidence is 0.54 because it's split.

## Reading a Score

Let's look at how the score changes with different inputs, using the same question and levels as above:

| State | `score` | `confidence` | Level 0 | Level 1 | Level 2 |
| --- | --- | --- | --- | --- | --- |
| The export button is misaligned by a few pixels on the settings page. | 0.0 | 1.0 | 1.0 | 0.0 | 0.0 |
| The PDF export button does nothing when clicked. I can still export to CSV and convert it myself, but that takes ages. | 1.0 | 1.0 | 0.0 | 1.0 | 0.0 |
| Export to PDF fails with a spinner that never finishes. Some of our team say CSV export still works for them, others say it fails too. | 1.12 | 0.81 | 0.0 | 0.88 | 0.12 |
| The export button crashes the settings page in Safari. It works in Chrome, but a few of our customers only use Safari. | 1.3 | 0.54 | 0.0 | 0.7 | 0.3 |
| Nobody on our team can log in since this morning. We get a 500 error on every attempt. | 2.0 | 1.0 | 0.0 | 0.0 | 1.0 |

In these examples, confidence 1.0 means the returned distribution puts all its probability on one level. This describes the model's answer, not a guarantee that the answer is correct.

The score is a probability-weighted mean of the level numbers. In the third and fourth examples, probability is split between levels 1 and 2. More weight on level 2 raises the score. It does not measure the fraction of customers without a workaround.

Different distributions can produce the same score. A score of 1.0 can mean all probability is on level 1, or half is on each of levels 0 and 2. Read `probabilities` and `confidence` alongside the score to distinguish these cases.

Low confidence on a Score usually means one of three things: the levels overlap for this state, the question is measuring more than one thing, or the state doesn't say enough to place it.

## Writing good levels

Describe situations, not degrees. "Broken or degraded feature, but workaround exists" gives the model something to match the state against. "Moderately severe" doesn't.

Every level is evaluated separately. The model doesn't see a level's number or its neighbours, so "worse than the previous level" means nothing to it, and numbers in the descriptions or the instructions don't help. Here is what happens when the levels are only numbers, on the misaligned-button report from the table above:

```
instructions: "Rate severity from 0 to 2, where 2 is worst"
criteria: ["0", "1", "2"]
→ score 0.57, confidence 0.35, probabilities 0: 0.43, 1: 0.57, 2: 0.0
```

The same report with the three descriptive levels scores 0.0 at confidence 1.0. With numbers only, the model has nothing to match against and splits the probability between 0 and 1.

Use as many levels as you can describe distinctly, up to 10. Three is fine. Don't add levels you can't describe distinctly.

Keep each Score question to one dimension. If a description says "punctual and smart and experienced", the question is measuring three things, and an input that is high on one and low on another can't be placed. Confidence drops and the score means less. Split it into one Score question per thing and combine them in code.

If there is no in-between at all, and the answer is one of a few discrete categories, use a [Choice](https://docs.typesafe.ai/primitives/choice) instead, or split the question into several [Noul](https://docs.typesafe.ai/primitives/noul) questions.

> This page also covers splitting a complex judgment into several weighted Score questions (the [Composite scoring](https://docs.typesafe.ai/patterns/composite-scoring) pattern) and structured level descriptions with worked examples — trimmed here since they're use-case material for a later lesson, not today's fundamentals. Read the full page at [docs.typesafe.ai/primitives/score](https://docs.typesafe.ai/primitives/score) any time.

</details>

## Score, in your own words

`Choice` picks one option from a fixed set. `Score` is for when the answer isn't one option but a *position* — how severe, how frustrated, how experienced — described as an ordered series of levels you write yourself.

The model doesn't see the level numbers or compare levels to each other — it judges each level description independently against the state, then returns a probability for every level. `score` is just the probability-weighted average position: a score of 1.3 out of a 0–2 scale means "mostly level 1, with some weight on level 2", not some separately-measured value that happens to land at 1.3.

In [ ]:
import { client } from "../assets/typesafe-client.ts";
import { score } from "npm:@typesafe-ai/sdk";

const response = await client.systemOne({
  state: "The export button crashes the settings page in Safari. It works in Chrome, but a few of our customers only use Safari.",
  questions: {
    bugSeverity: score("How severe is the reported issue?", [
      "Cosmetic; no impact to functionality",
      "Broken or degraded feature, but workaround exists",
      "Blocking issue; no workaround exists",
    ]),
  },
});

console.log(response.answers.bugSeverity);

## What just happened

`response.answers.bugSeverity` comes back with four fields: `score` (the position), `confidence` (how peaked the distribution is), `legend` (level number → description), and `probabilities` (the full distribution). The primary source above shows this exact state scoring `1.3` at confidence `0.54` — worth checking whether your run lands close to that, since it's a live model call, not a fixed lookup.

**Try it yourself:** change the ticket text to something clearly cosmetic (a typo, a misaligned icon) or clearly blocking (nobody can log in). Predict the `score` and roughly how confident it'll be before you re-run the cell.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/primitives/noul (click to expand)</summary>

# Noul

> A Noul question asks the model to evaluate a yes/no question and return the probability that the answer is yes.

Use a Noul when the answer is yes or no. For example, does this message ask for a refund, does this resume mention distributed systems, does this comment contain personal data. If the answer is one of several options, use a [Choice](https://docs.typesafe.ai/primitives/choice). If it's a position on a spectrum, use a [Score](https://docs.typesafe.ai/primitives/score). [Choose a question type](https://docs.typesafe.ai/primitives#choose-a-question-type) compares all three.

A Noul answer is a single number, `noul`, the probability that the answer is yes.

## Writing a Noul question

A Noul question evaluates a single yes/no question (or statement). It is defined by its `instructions`: the yes/no question to evaluate. It's good practice to phrase it so a high probability means "yes", so that the returned answer is unambiguous in its meaning.

You can optionally add `criteria` with `true` and `false` descriptions to clarify what each outcome means, which can be helpful when the question itself has more nuance to explain. Try your Noul question prompts with and without criteria to see which works better in your use-case.

## Request

| Field | Required | Description |
| --- | --- | --- |
| `type` | Yes | Must be `"noul"`. |
| `instructions` | Yes | The yes/no question or statement to evaluate. |
| `criteria` | No | Optional `{ true, false }` descriptions clarifying what a yes and a no mean. |

```json
{
  "state": "I have asked three times now. Can I please just talk to a real person?",
  "questions": {
    "is_human_escalation": {
      "type": "noul",
      "instructions": "Is the customer asking for a human agent?"
    },
    "is_repeat_contact": {
      "type": "noul",
      "instructions": "Has the customer contacted support about this before?",
      "criteria": {
        "true": "Mentions a prior attempt, ticket, or that they have asked before",
        "false": "No sign of any previous contact"
      }
    }
  }
}
```

## Response

```json
{
  "model": "jev-latest",
  "answers": {
    "is_human_escalation": {
      "type": "noul",
      "noul": 0.99
    },
    "is_repeat_contact": {
      "type": "noul",
      "noul": 0.93
    }
  },
  "usage": {
    "input_tokens": 360,
    "output_tokens": 39
  }
}
```

`noul` ranges from 0 to 1, representing the probability that the answer is **yes**. Most often you will threshold it into a boolean when your code needs a hard decision.

## Noul does not return a separate confidence value

A value near 1 means a strong yes. A value near 0 means a strong no. A value near 0.5 gives yes and no similar probability.

For "Is the candidate strong in Python?", define what "strong" means. An unclear definition makes the probability hard to interpret. A value of 0.5 does not mean medium skill. Use a [Score](https://docs.typesafe.ai/primitives/score) to measure skill along defined levels.

## Example questions

```
"Is the customer requesting a refund?"
"Does this resume mention experience with distributed systems?"
"Does the message contain personally identifiable information?"
"Does the room have a minifridge?"
```

## Tips and advanced usage

- **Phrasing.** Beyond a plain question, you can phrase the instruction as a statement for the model to evaluate for truthfulness. For "the customer is requesting a refund", a value near 1 means the statement is true. Try both phrasings with your own data to see what works best.
- **Optional `criteria`.** The instruction is enough for most Noul questions, but when the boundary between yes and no is subtle, pass `criteria` with `true` and `false` descriptions to pin down what each outcome means — as shown in the request example above.

</details>

## Noul, in your own words

`Noul` is the simplest primitive: one yes/no question, one number back. No options to define, no levels to write — just `noul`, the probability the answer is yes.

The one thing worth getting right: **Noul has no separate `confidence` field.** `0.5` isn't "medium confidence" — it means the model gives yes and no equal probability. If you find yourself wanting a "how sure are you" reading alongside a yes/no answer, that's usually a sign the question is really a `Score` in disguise.

In [ ]:
import { client } from "../assets/typesafe-client.ts";
import { noul } from "npm:@typesafe-ai/sdk";

const response = await client.systemOne({
  state: "I have asked three times now. Can I please just talk to a real person?",
  questions: {
    isHumanEscalation: noul("Is the customer asking for a human agent?"),
    isRepeatContact: noul(
      "Has the customer contacted support about this before?",
      {
        true: "Mentions a prior attempt, ticket, or that they have asked before",
        false: "No sign of any previous contact",
      },
    ),
  },
});

console.log(response.answers.isHumanEscalation);
console.log(response.answers.isRepeatContact);

## What just happened

Two independent yes/no judgments against the same state, each returning just a `noul` value — no `confidence`, no `probabilities`, no `legend`. `isHumanEscalation` should land close to 1 (the message is a direct, explicit ask); `isRepeatContact` is more interesting, since the model has to infer "three times now" implies prior contact without being told that directly.

**Try it yourself:** rewrite the message so it's genuinely ambiguous about whether this is a repeat contact (remove "three times now" entirely). Predict whether `isRepeatContact` moves toward 0.5 before you re-run.

<details>
  <summary><strong>Primary source (excerpt):</strong> docs.typesafe.ai/primitives — "Choose a question type" (click to expand)</summary>

| Type | What it answers | Returns |
| --- | --- | --- |
| [Choice](https://docs.typesafe.ai/primitives/choice) | Which of these options? | `choice`, `probabilities`, `confidence` |
| [Score](https://docs.typesafe.ai/primitives/score) | Which level? | `score`, `legend`, `probabilities`, `confidence` |
| [Noul](https://docs.typesafe.ai/primitives/noul) | Is this true? | `noul` (0 to 1) |

## Choose a question type

Pick the type that matches the shape of the answer you need.

- **Choice** fits when the answer is one of a known set of options with no order between them: routing a ticket to a department, classifying a document type, detecting a programming language. Give the full list of options, and add an `other` or `none of the above` option when the list might not cover every input.
- **Score** fits when the answer falls on a spectrum and you can describe what each point on that spectrum means: bug severity, customer frustration, skill level. The levels are yours to define, and the model returns a position along them.
- **Noul** fits a clean yes/no question where the probability itself is the useful signal: does this message report a bug, is the customer requesting a refund, does the resume mention distributed systems.

> **Note:** Use Noul for a yes/no judgment and Score to measure a position on a spectrum. "Is this candidate strong in Python?" needs a clear definition of "strong". A Noul value of 0.5 means the model gives yes and no equal probability. It does not mean the candidate has a medium skill level. An unclear definition makes that probability hard to interpret.
>
> If you want to measure skill level, use a Score with defined levels, such as no experience, some familiarity, daily use, and deep expertise. If you need a yes/no decision, define the condition clearly, such as "Does the resume state that the candidate has used Python at work?"

If two types both seem to fit, prefer the one whose answer your code can act on directly. A Choice between `refund`, `rebook`, and `information` maps straight onto three code paths. A Score of customer frustration maps onto a threshold. A Noul maps onto an `if`.

</details>

## Why picking the right one matters

The three primitives aren't interchangeable skins on the same idea — they shape what your code can do with the answer:

- **Choice** → a fixed set of code paths (`switch` on the result)
- **Score** → a position you threshold or rank against (`if score > 1.5`)
- **Noul** → a boolean decision (`if isRefundRequest`)

Picking the wrong one costs you downstream. A `Noul` where you actually needed a `Score` throws away the "how much" signal and leaves you re-deriving it from a single probability that was never designed to carry it. A `Choice` where you needed a `Score` forces the model to pick a discrete bucket instead of reporting a position that could legitimately sit between two of your buckets.

## Next up

Next lesson: composing multiple questions in one request — the Speculative fan-out pattern — and what "ask everything you might need, let code decide what applies" actually buys you.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.